# Box Office Bomb Data Pipeline Assignment

**Total Marks: 20**

This notebook implements an end-to-end pipeline to:
1. Scrape box office bomb data from Wikipedia
2. Validate & clean data using Pydantic
3. Enrich with OMDb API data
4. Perform consistency checks
5. Create final categorized dataset

## Setup and Imports

In [ ]:
# Import required libraries
import pandas as pd
import requests
from bs4 import BeautifulSoup
from pydantic import BaseModel, field_validator, ValidationError
import re
import time
from pathlib import Path
from typing import Optional, Dict, Any

## Task 1: Scrape the "Bombs" Table (4 Marks)

Extract raw data from the Wikipedia HTML file:
- Film Title (with symbols like § and †)
- Year
- Net production budget (may contain ranges like "$100–160")
- Estimated loss (Nominal column)

Note:
- You need to extract the entire raw string with the symbols, references, etc along with the titles.
- You must handle the nested headers in the Wikipedia table (Budget and Loss columns have sub-headers).

In [ ]:
def scrape_box_office_bombs():
    """
    Scrape the box office bombs table from Wikipedia.
    Returns a list of dictionaries with raw extracted data.
    """
    url = "https://en.wikipedia.org/wiki/List_of_biggest_box-office_bombs"
    headers = {'User-Agent': 'Mozilla/5.0 '}
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")

    # Find ALL wikitable tables and inspect them
    tables = soup.find_all("table", class_="wikitable")
    table = None

    for t in tables:
        caption = t.find("caption")
        if caption and any(phrase in caption.get_text() for phrase in ["bombs", "Biggest"]):
            table = t
            break

    # Alternative: Find table by looking for specific content
    if table is None:
        for t in tables:
            if t.find("th", string=re.compile(r"Budget|Net production", re.I)):
                table = t
                break

    if table is None:
        print("No suitable table found. Available tables:")
        for i, t in enumerate(tables[:3]):
            print(f"Table {i}: {t.find('caption')}")
        return []

    raw_data = []
    rows = table.find_all("tr")

    # Skip header rows (first 2-3 rows)
    for row in rows[2:]:
        cols = row.find_all(["td", "th"])
        if len(cols) < 4:
            continue

        # Title (keep raw with symbols)
        raw_title = cols[0].get_text(strip=True)

        # Year
        raw_year = cols[1].get_text(strip=True)

        # Budget
        raw_budget = ""
        if len(cols) > 2:
            raw_budget = cols[2].get_text(strip=True)

        # Loss
        raw_loss = ""
        if len(cols) > 4:
            raw_loss = cols[4].get_text(strip=True)

        if raw_title and raw_year:
            raw_data.append({
                "raw_title": raw_title,
                "raw_year": raw_year,
                "raw_budget": raw_budget,
                "raw_loss": raw_loss
            })

    return raw_data


In [ ]:
# Test the scraping function
raw_movies = scrape_box_office_bombs()
print(f"Scraped {len(raw_movies)} movies")
print("\nLast 15 raw entries:")
for i, movie in enumerate(raw_movies[:]):
    print(f"\n{i+1}. {movie}")

Scraped 139 movies

Last 15 raw entries:

1. {'raw_title': 'The 13th Warrior', 'raw_year': '1999', 'raw_budget': '$100–160', 'raw_loss': '$69–129'}

2. {'raw_title': 'The 355', 'raw_year': '2022', 'raw_budget': '$40–75', 'raw_loss': '$93'}

3. {'raw_title': '47 Ronin', 'raw_year': '2013', 'raw_budget': '$175–225', 'raw_loss': '$96'}

4. {'raw_title': 'The Adventures of Baron Munchausen', 'raw_year': '1988', 'raw_budget': '$46.6', 'raw_loss': '$38.5'}

5. {'raw_title': 'The Adventures of Pluto Nash', 'raw_year': '2002', 'raw_budget': '$100', 'raw_loss': '$96'}

6. {'raw_title': 'The Adventures of Rocky & Bullwinkle', 'raw_year': '2000', 'raw_budget': '$76–98.6', 'raw_loss': '$63.5'}

7. {'raw_title': 'The Alamo', 'raw_year': '2004', 'raw_budget': '$107', 'raw_loss': '$94'}

8. {'raw_title': 'Alexander', 'raw_year': '2004', 'raw_budget': '$155', 'raw_loss': '$71'}

9. {'raw_title': 'Ali', 'raw_year': '2001', 'raw_budget': '$107', 'raw_loss': '$63'}

10. {'raw_title': 'Allied', 'raw_year'

## Task 2: Pydantic Data Parsing & Validation (6 Marks)

Create a Pydantic model that:
- Cleans titles (removes §, †, and footnotes like [nb 2], [1])
- Parses numeric values (handles ranges, currency symbols)
- Validates year as integer

In [ ]:
class MovieData(BaseModel):
    title: str
    year: int
    budget_millions: float
    loss_millions: float

    @field_validator('title', mode='before')
    @classmethod
    def clean_title(cls, v):
        if not v:
            return "" # if no title, return empty string
        # Remove special symbols
        v = v.replace('§', '').replace('†', '')
        # Remove footnotes [nb 1], [1], etc.
        v = re.sub(r'\[\w+( \w+)*\]', '', v)
        # Remove extra whitespaces
        return v.strip()

    @field_validator('year', mode='before')
    @classmethod
    def validate_year(cls, v):
        if not v:
            raise ValueError("Year cannot be empty")
        # Extract digits only
        year_str = re.sub(r'\D', '', str(v))
        if not year_str:
            raise ValueError("No valid year found")
        return int(year_str)

    @field_validator('budget_millions', mode='before')
    @classmethod
    def parse_budget(cls, v):
        return cls._parse_numeric_value(v)

    @field_validator('loss_millions', mode='before')
    @classmethod
    def parse_loss(cls, v):
        return cls._parse_numeric_value(v)

    @staticmethod
    def _parse_numeric_value(v):
        if not v or pd.isna(v): # missing value
            return 0.0

        v = str(v)
        # Remove currency symbols and references
        v = re.sub(r'[\$,€£]', '', v)
        v = re.sub(r'\[\d+\]', '', v)
        v = re.sub(r'\[nb \d+\]', '', v)

        # Handle ranges like "100–160" or "100-160"
        if '–' in v or '—' in v or '-' in v:
            # Extract numbers around dash
            numbers = re.findall(r'(\d+(?:\.\d+)?)', v)
            if len(numbers) >= 2:
                return float(sum(float(n) for n in numbers[:2])) / 2
            elif numbers:
                return float(numbers[0])

        # Single number, when dash is not there
        numbers = re.findall(r'(\d+(?:\.\d+)?)', v)
        if numbers:
            return float(numbers[0])

        return 0.0

In [ ]:
# Validate data
validated_movies = []
failed_validations = []

for raw_movie in raw_movies:
    try:
        movie = MovieData(
            title=raw_movie['raw_title'],
            year=raw_movie['raw_year'],
            budget_millions=raw_movie['raw_budget'],
            loss_millions=raw_movie['raw_loss']
        )
        validated_movies.append(movie)
    except (ValidationError, ValueError) as e:
        failed_validations.append({
            'raw_data': raw_movie,
            'error': str(e)
        })

print(f"\nValidation Results:")
print(f"Successfully validated: {len(validated_movies)} movies")
print(f"Failed validations: {len(failed_validations)} movies")

print("\nValidated movies:")
for movie in validated_movies[:]:
    print(movie.model_dump())



Validation Results:
Successfully validated: 139 movies
Failed validations: 0 movies

Validated movies:
{'title': 'The 13th Warrior', 'year': 1999, 'budget_millions': 130.0, 'loss_millions': 99.0}
{'title': 'The 355', 'year': 2022, 'budget_millions': 57.5, 'loss_millions': 93.0}
{'title': '47 Ronin', 'year': 2013, 'budget_millions': 200.0, 'loss_millions': 96.0}
{'title': 'The Adventures of Baron Munchausen', 'year': 1988, 'budget_millions': 46.6, 'loss_millions': 38.5}
{'title': 'The Adventures of Pluto Nash', 'year': 2002, 'budget_millions': 100.0, 'loss_millions': 96.0}
{'title': 'The Adventures of Rocky & Bullwinkle', 'year': 2000, 'budget_millions': 87.3, 'loss_millions': 63.5}
{'title': 'The Alamo', 'year': 2004, 'budget_millions': 107.0, 'loss_millions': 94.0}
{'title': 'Alexander', 'year': 2004, 'budget_millions': 155.0, 'loss_millions': 71.0}
{'title': 'Ali', 'year': 2001, 'budget_millions': 107.0, 'loss_millions': 63.0}
{'title': 'Allied', 'year': 2016, 'budget_millions': 85.

## Task 3: Enrich with OMDb Data (4 Marks)

Query OMDb API for each movie to get:
- Plot
- Metascore
- IMDb Rating
- Director
- Language

Handle API failures (Response='False') or missing fields ('N/A') gracefully by storing them as None. Do not delete the row.

In [ ]:
OMDB_API_KEY = "4f870784"  # Get free key from http://www.omdbapi.com/apikey.aspx
OMDB_BASE_URL = "http://www.omdbapi.com/"


def _to_int_or_none(v: str | None) -> int | None:
    if v is None or v == "N/A":
        return None
    try:
        return int(v)
    except ValueError:
        return None

def query_omdb(title: str, year: int) -> Dict[str, Any]:
    """
    Query OMDb API for movie metadata.
    """
    params = {
        'apikey': OMDB_API_KEY,
        't': title,
        'y': year,
        'plot': 'short'
    }
    result = {
            'plot': None,
            'metascore': None,
            'imdb_rating': None,
            'director': None,
            'language': None,
            "omdb_year": None,

    }

    try:
        response = requests.get(OMDB_BASE_URL, params=params, timeout=10)
        data = response.json()

        if data.get('Response') == 'False':
          return result


        # Extract fields, convert N/A to None
        result = {
            'plot': data.get('Plot') if data.get('Plot') != 'N/A' else None,
            'metascore': float(data.get('Metascore', 0)) if data.get('Metascore') != 'N/A' else None,
            'imdb_rating': float(data.get('imdbRating', 0)) if data.get('imdbRating') != 'N/A' else None,
            'director': data.get('Director') if data.get('Director') != 'N/A' else None,
            'language': data.get('Language') if data.get('Language') != 'N/A' else None,
            "omdb_year": _to_int_or_none(data.get("Year")),
        }
        return result

    except Exception:
        return result



In [ ]:
enriched_data = []
print("Querying OMDb API...")

for i, movie in enumerate(validated_movies[:]):
    omdb_data = query_omdb(movie.title, movie.year)
    enriched = movie.model_dump()
    enriched.update(omdb_data)
    enriched_data.append(enriched)

    if i % 10 == 0:
        print(f"Processed {i+1}/{len(validated_movies)}")
    time.sleep(1)  # Rate limiting

print(f"\nEnriched {len(enriched_data)} movies with OMDb data")
print("\nEnriched Data:", enriched_data[:])


Querying OMDb API...
Processed 1/139
Processed 11/139
Processed 21/139
Processed 31/139
Processed 41/139
Processed 51/139
Processed 61/139
Processed 71/139
Processed 81/139
Processed 91/139
Processed 101/139
Processed 111/139
Processed 121/139
Processed 131/139

Enriched 139 movies with OMDb data

Enriched Data: [{'title': 'The 13th Warrior', 'year': 1999, 'budget_millions': 130.0, 'loss_millions': 99.0, 'plot': 'A man, having fallen in love with the wrong woman, is sent by the sultan himself on a diplomatic mission to a distant land as an ambassador. Stopping at a Viking village port to restock on supplies, he finds himself unwittingly em...', 'metascore': 42.0, 'imdb_rating': 6.6, 'director': 'John McTiernan', 'language': 'English, Latin, Swedish, Norse, Old, Danish, Arabic', 'omdb_year': 1999}, {'title': 'The 355', 'year': 2022, 'budget_millions': 57.5, 'loss_millions': 93.0, 'plot': "When a top-secret weapon falls into mercenary hands, a wild-card C.I.A. agent joins forces with thr

OMDB Data:

In [ ]:
for i, movie in enumerate(validated_movies[:]):
    omdb_data = query_omdb(movie.title, movie.year)
    print("OMDB Data:")
    print(omdb_data)


OMDB Data:
{'plot': 'A man, having fallen in love with the wrong woman, is sent by the sultan himself on a diplomatic mission to a distant land as an ambassador. Stopping at a Viking village port to restock on supplies, he finds himself unwittingly em...', 'metascore': 42.0, 'imdb_rating': 6.6, 'director': 'John McTiernan', 'language': 'English, Latin, Swedish, Norse, Old, Danish, Arabic', 'omdb_year': 1999}
OMDB Data:
{'plot': "When a top-secret weapon falls into mercenary hands, a wild-card C.I.A. agent joins forces with three international agents on a mission to retrieve it, while staying a step ahead of a mysterious woman who's tracking their every move.", 'metascore': 40.0, 'imdb_rating': 5.6, 'director': 'Simon Kinberg', 'language': 'English, Chinese, Spanish, French, German, Arabic, Romanian, Hindi, Sign ', 'omdb_year': 2022}
OMDB Data:
{'plot': 'A band of samurai sets out to avenge the death and dishonor of their master at the hands of a ruthless shogun.', 'metascore': 28.0, 'i

## Task 4: Data Consistency Check (2 Marks)

Compare Wikipedia year with OMDb year:
- "Verified": Years match (±1 year tolerance)
- "Mismatch": Years differ by >1
- "Not Found": OMDb returned no data

In [ ]:

def determine_match_status(wiki_year: int, omdb_year: Optional[int]) -> str:
    """
    Determine the match status between Wikipedia and OMDb years.
    """
    if omdb_year is None:
        # print()
        return "Not Found"

    year_diff = abs(wiki_year - omdb_year)
    if year_diff <= 1:
        return "Verified"
    return "Mismatch"

# Add match status
for entry in enriched_data:
    entry['match_status'] = determine_match_status(entry['year'], entry.get('omdb_year'))

# Show results
df_temp = pd.DataFrame(enriched_data)
print("Match Status Distribution:")
print(df_temp['match_status'].value_counts())

mismatches = df_temp[df_temp['match_status'] == 'Mismatch']
# print(mismatches)
if len(mismatches) > 0:
    print(f"\nSample Mismatches (showing up to 5):")
    print(mismatches[['title', 'year', 'omdb_year', 'match_status']].head())
else:
    print("No mismatches found")

Match Status Distribution:
match_status
Verified     137
Not Found      2
Name: count, dtype: int64
No mismatches found


## Task 5: Final Dataset & Categorization (4 Marks)

Create final DataFrame with:
- Loss_Category based on estimated loss:
  - "Catastrophic": Loss ≥ \$100M

  - "Severe": Loss between \$50M and \$100M

  - "Moderate": Loss < \$50M
- Save to `box_office_failures.csv`

Required columns: Title, Year, Director, Language, Budget_Millions, Loss_Millions, Loss_Category, IMDb_Rating, Metascore, Match_Status

In [ ]:
def categorize_loss(loss_millions: float) -> str:
    """
    Categorize the financial loss into tiers.
    """
    if loss_millions >= 100:
        return "Catastrophic"
    elif loss_millions >= 50:
        return "Severe"
    return "Moderate"

# Create final DataFrame
df_final = pd.DataFrame(enriched_data)

# Add loss category
df_final['Loss_Category'] = df_final['loss_millions'].apply(categorize_loss)

# Select and rename required columns
final_columns = {
    'title': 'Title',
    'year': 'Year',
    'director': 'Director',
    'language': 'Language',
    'budget_millions': 'Budget_Millions',
    'loss_millions': 'Loss_Millions',
    'Loss_Category': 'Loss_Category',
    'imdb_rating': 'IMDb_Rating',
    'metascore': 'Metascore',
    'match_status': 'Match_Status'
}

df_final = df_final.rename(columns=final_columns)[list(final_columns.values())]

# Display summary
print("Final Dataset Summary:")
print(f"Total movies: {len(df_final)}")
print(f"\nLoss Category Distribution:")
print(df_final['Loss_Category'].value_counts())
print(f"\nBasic Statistics:")
print(df_final[['Budget_Millions', 'Loss_Millions', 'IMDb_Rating', 'Metascore']].describe())

print(f"\nFirst 10 rows of final dataset:")
df_final

Final Dataset Summary:
Total movies: 139

Loss Category Distribution:
Loss_Category
Severe          85
Catastrophic    46
Moderate         8
Name: count, dtype: int64

Basic Statistics:
       Budget_Millions  Loss_Millions  IMDb_Rating   Metascore
count       139.000000     139.000000   137.000000  134.000000
mean        129.552518      91.917266     5.816058   46.529851
std          58.126686      32.407538     1.098276   15.359693
min          17.000000      10.800000     2.100000    9.000000
25%          81.250000      70.550000     5.400000   37.000000
50%         117.500000      85.000000     6.000000   46.000000
75%         175.000000     108.500000     6.600000   56.750000
max         326.000000     218.000000     8.000000   85.000000

First 10 rows of final dataset:


,Title,Year,Director,Language,Budget_Millions,Loss_Millions,Loss_Category,IMDb_Rating,Metascore,Match_Status
0,The 13th Warrior,1999,John McTiernan,"English, Latin, Swedish, Norse, Old, Danish, A...",130.0,99.0,Severe,6.6,42.0,Verified
1,The 355,2022,Simon Kinberg,"English, Chinese, Spanish, French, German, Ara...",57.5,93.0,Severe,5.6,40.0,Verified
2,47 Ronin,2013,Carl Rinsch,"English, Japanese",200.0,96.0,Severe,6.2,28.0,Verified
3,The Adventures of Baron Munchausen,1988,Terry Gilliam,English,46.6,38.5,Moderate,7.1,69.0,Verified
4,The Adventures of Pluto Nash,2002,Ron Underwood,English,100.0,96.0,Severe,3.9,12.0,Verified
...,...,...,...,...,...,...,...,...,...,...
134,The Wolfman,2010,Joe Johnston,"English, Romany, Romanian, Ukrainian",150.0,76.0,Severe,5.9,43.0,Verified
135,Wonder Woman 1984,2020,Patty Jenkins,"English, Arabic, Russian, Mandarin",200.0,117.5,Catastrophic,5.3,60.0,Verified
136,A Wrinkle in Time,2018,Ava DuVernay,English,125.0,130.6,Catastrophic,4.3,53.0,Verified
137,xXx: State of the Union,2005,Lee Tamahori,English,113.1,78.0,Severe,4.5,37.0,Verified


In [ ]:
# Save to CSV with NaN explicitly written
output_path = 'box_office_failures.csv'
df_final.to_csv(output_path, index=False, na_rep='NaN')

print(f"\n✓ Dataset saved to: {output_path}")
print(f"✓ Total records: {len(df_final)}")



✓ Dataset saved to: box_office_failures.csv
✓ Total records: 139


## Additional Analysis (Optional)

In [ ]:
# Show some interesting insights
print("Top 10 Biggest Box Office Bombs (by loss):")
print(df_final.nlargest(10, 'Loss_Millions')[['Title', 'Year', 'Director', 'Loss_Millions', 'Loss_Category']])

print("\n" + "="*60)
print("\nMovies with Lowest IMDb Ratings:")
lowest_rated = df_final[df_final['IMDb_Rating'].notna()].nsmallest(5, 'IMDb_Rating')
print(lowest_rated[['Title', 'Year', 'Director', 'IMDb_Rating', 'Metascore', 'Loss_Millions']])

print("\n" + "="*60)
print("\nAverage Loss by Category:")
print(df_final.groupby('Loss_Category')['Loss_Millions'].agg(['mean', 'count']))

print("\n" + "="*60)
print("\nTop 5 Most Common Directors in Box Office Bombs:")
print(df_final['Director'].value_counts().head())

Top 10 Biggest Box Office Bombs (by loss):
                   Title  Year              Director  Loss_Millions  \
80           The Marvels  2023           Nia DaCosta          218.0   
116        Strange World  2022  Don Hall, Qui Nguyen          187.5   
77       The Lone Ranger  2013        Gore Verbinski          175.0   
86        Mortal Engines  2018      Christian Rivers          174.8   
128          Turning Red  2022             Domee Shi          173.0   
67           John Carter  2012        Andrew Stanton          156.0   
42             The Flash  2023       Andy Muschietti          155.0   
69         Jungle Cruise  2021    Jaume Collet-Serra          150.0   
68   Joker: Folie à Deux  2024         Todd Phillips          144.3   
87                 Mulan  2020             Niki Caro          141.0   

    Loss_Category  
80   Catastrophic  
116  Catastrophic  
77   Catastrophic  
86   Catastrophic  
128  Catastrophic  
67   Catastrophic  
42   Catastrophic  
69   Catastroph

In [ ]:
print(df_final.groupby('Director')['Loss_Millions'].sum().idxmax())

Jaume Collet-Serra


In [ ]:
df
